# Ransomware Detection Using Hardware Telemetry
## Polished AI/ML Training, EDA, Validation & TinyML Export

This notebook preserves the **original training pipeline and model behavior** while restructuring it into clear sections and adding rigorous exploratory data analysis.

### What is preserved
- Same 8 input features
- Same `55:45` class distribution
- Same stratified train/validation/test split
- Same random seed (`42`)
- Same neural-network architecture: `64 → 32 → 1`
- Same normalization strategy
- Same dropout (`0.30`) and L2 regularization (`1e-4`)
- Same Adam learning rate (`0.001`)
- Same class weighting
- Same early stopping strategy
- Same decision threshold (`0.45`)
- Same 5-fold cross-validation
- Same float16 TFLite conversion
- Same C-header generation

### Added analysis
- Data-quality audit
- Class-distribution analysis
- Descriptive statistics
- Feature histograms
- Boxplots
- Correlation heatmap
- Class-wise feature means
- Single-feature ROC-AUC separability test
- PCA visualization
- Feature redundancy analysis
- Training-curve analysis
- ROC and Precision-Recall curves
- Confusion matrix
- Threshold sensitivity analysis
- Error analysis
- Full-test-set Keras ↔ TFLite parity check

> **Important:** EDA is analysis-only. It does not modify the feature values or change the model-training pipeline.

## 1. Imports & Reproducibility

In [ ]:
import os
import sys
import time
import json
import random
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, regularizers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

## 2. Configuration

In [ ]:
FEATURES = [
    "USB_V", "USB_A", "USB_W", "Int_Accel",
    "Ext_Accel", "Net_Traffic", "RF_Density", "Sys_Temp"
]
LABEL = "Label"

LABEL_MAP = {"No": 0, "Yes": 1}
LABEL_NAMES = ["Normal (No)", "Ransomware (Yes)"]

BATCH_SIZE = 32
MAX_EPOCHS = 150
LEARNING_RATE = 0.001
DROPOUT_RATE = 0.30
L2_LAMBDA = 1e-4
PATIENCE = 15

VAL_SPLIT = 0.15
TEST_SPLIT = 0.15
RANDOM_SEED = 42

# Preserved from the original code to keep the same deployed behavior.
DECISION_THRESHOLD = 0.45

MODEL_H_OUT = "ransomware_model_new.h"
MODEL_TFLITE = "ransomware_model_new.tflite"
REPORT_FILE = "training_report.txt"
PLOT_FILE = "training_plots.png"

def find_dataset():
    candidates = [
        Path("/kaggle/input/datasets/swastiksingh123/ransomware-hardware/ransomware_dataset_3k_55_45 (1).csv"),
        Path("/kaggle/input/datasets/swastiksingh123/ransom/ransomware_dataset_3k_55_45 (3).csv"),
        Path("/kaggle/working/ransomware_dataset_3k_55_45.csv"),
        Path("/mnt/data/ransomware_dataset_3k_55_45.csv"),
        Path("ransomware_dataset_3k_55_45.csv"),
    ]

    for p in candidates:
        if p.exists():
            return p

    for root in [Path("/kaggle/input"), Path("/mnt/data"), Path(".")]:
        if root.exists():
            csvs = list(root.rglob("*.csv"))
            ransomware_csvs = [p for p in csvs if "ransom" in p.name.lower()]
            if ransomware_csvs:
                return ransomware_csvs[0]
            if csvs:
                return csvs[0]

    raise FileNotFoundError(
        "Dataset not found. Set CSV_FILE manually to the ransomware CSV path."
    )

CSV_FILE = find_dataset()
print("Dataset:", CSV_FILE)

## 3. Load & Encode Dataset

In [ ]:
df = pd.read_csv(CSV_FILE)

missing_columns = [c for c in FEATURES + [LABEL] if c not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Keep the original label mapping.
if df[LABEL].dtype == object:
    unknown = set(df[LABEL].dropna().unique()) - set(LABEL_MAP.keys())
    if unknown:
        raise ValueError(f"Unexpected labels: {unknown}. Expected No/Yes.")
    df[LABEL] = df[LABEL].map(LABEL_MAP)

df[LABEL] = df[LABEL].astype(int)

for feature in FEATURES:
    df[feature] = pd.to_numeric(df[feature], errors="coerce")

n_normal = int((df[LABEL] == 0).sum())
n_ransom = int((df[LABEL] == 1).sum())

print(f"Rows             : {len(df):,}")
print(f"Columns          : {len(df.columns)}")
print(f"Normal           : {n_normal:,} ({100*n_normal/len(df):.2f}%)")
print(f"Ransomware       : {n_ransom:,} ({100*n_ransom/len(df):.2f}%)")
print(f"Missing values   : {df[FEATURES+[LABEL]].isna().sum().sum()}")
print(f"Duplicate rows   : {df.duplicated().sum()}")

display(df.head())

## 4. Data Quality Audit

This section checks the integrity of the dataset without modifying it.  
If missing values or duplicates exist, they should be investigated rather than silently removed.

In [ ]:
quality_report = pd.DataFrame({
    "dtype": df[FEATURES + [LABEL]].dtypes.astype(str),
    "missing": df[FEATURES + [LABEL]].isna().sum(),
    "unique_values": df[FEATURES + [LABEL]].nunique()
})

display(quality_report)

if df[FEATURES + [LABEL]].isna().any().any():
    print("WARNING: Missing values exist. The original pipeline assumes complete records.")

if df.duplicated().sum() > 0:
    print("WARNING: Duplicate rows exist. Review whether they are expected.")

## 5. Class Distribution

In [ ]:
class_counts = df[LABEL].value_counts().sort_index()
class_percent = 100 * class_counts / class_counts.sum()

class_table = pd.DataFrame({
    "Class": LABEL_NAMES,
    "Count": [class_counts.get(0, 0), class_counts.get(1, 0)],
    "Percentage": [class_percent.get(0, 0), class_percent.get(1, 0)]
})
display(class_table)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(LABEL_NAMES, class_table["Count"])
ax.set_title("Class Distribution")
ax.set_ylabel("Number of samples")
for i, value in enumerate(class_table["Count"]):
    ax.text(i, value + max(class_table["Count"]) * 0.01, str(value), ha="center")
plt.tight_layout()
plt.show()

## 6. Descriptive Statistics by Class

In [ ]:
stats_by_class = df.groupby(LABEL)[FEATURES].agg(
    ["min", "max", "mean", "median", "std"]
)
display(stats_by_class)

mean_comparison = df.groupby(LABEL)[FEATURES].mean().T
mean_comparison.columns = ["Normal_mean", "Ransomware_mean"]
mean_comparison["Absolute_difference"] = (
    mean_comparison["Ransomware_mean"] - mean_comparison["Normal_mean"]
).abs()

display(mean_comparison.sort_values("Absolute_difference", ascending=False))

## 7. Feature Distribution EDA

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for ax, feature in zip(axes.ravel(), FEATURES):
    normal = df.loc[df[LABEL] == 0, feature]
    ransomware = df.loc[df[LABEL] == 1, feature]

    ax.hist(normal, bins=30, alpha=0.55, density=True, label="Normal")
    ax.hist(ransomware, bins=30, alpha=0.55, density=True, label="Ransomware")
    ax.set_title(feature)
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)

plt.suptitle("Feature Distributions by Class", y=1.02)
plt.tight_layout()
plt.show()

## 8. Boxplot Analysis

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))

for ax, feature in zip(axes.ravel(), FEATURES):
    normal = df.loc[df[LABEL] == 0, feature].dropna()
    ransomware = df.loc[df[LABEL] == 1, feature].dropna()

    ax.boxplot(
        [normal, ransomware],
        labels=["Normal", "Ransomware"],
        showfliers=True
    )
    ax.set_title(feature)
    ax.tick_params(axis="x", rotation=15)

plt.suptitle("Class-wise Feature Boxplots", y=1.02)
plt.tight_layout()
plt.show()

## 9. Correlation & Redundancy Analysis

Highly correlated features may carry overlapping information. This is especially relevant for `USB_V`, `USB_A`, and `USB_W`, since power is physically related to voltage and current.

In [ ]:
corr = df[FEATURES].corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr.values, vmin=-1, vmax=1)

ax.set_xticks(range(len(FEATURES)))
ax.set_xticklabels(FEATURES, rotation=45, ha="right")
ax.set_yticks(range(len(FEATURES)))
ax.set_yticklabels(FEATURES)
ax.set_title("Feature Correlation Matrix")

for i in range(len(FEATURES)):
    for j in range(len(FEATURES)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

display(corr.round(4))

print("\nUSB voltage/current/power sub-correlation:")
display(df[["USB_V", "USB_A", "USB_W"]].corr().round(4))

## 10. Single-Feature Separability Analysis

A strong ML model is more meaningful when multiple features are required together.  
This diagnostic checks how well each individual feature alone separates the two classes.

In [ ]:
single_feature_rows = []

for feature in FEATURES:
    raw_auc = roc_auc_score(df[LABEL], df[feature])
    if raw_auc >= 0.5:
        best_auc = raw_auc
        direction = "higher → ransomware"
    else:
        best_auc = 1 - raw_auc
        direction = "lower → ransomware"

    single_feature_rows.append({
        "Feature": feature,
        "Best single-feature ROC-AUC": best_auc,
        "Direction": direction
    })

single_feature_auc = (
    pd.DataFrame(single_feature_rows)
    .sort_values("Best single-feature ROC-AUC", ascending=False)
    .reset_index(drop=True)
)

display(single_feature_auc)

best_single_auc = single_feature_auc.iloc[0]["Best single-feature ROC-AUC"]

if best_single_auc >= 0.98:
    print(
        "WARNING: At least one feature almost perfectly separates the classes. "
        "This means the current dataset is an easy classification problem."
    )
elif best_single_auc >= 0.90:
    print("CAUTION: Strong single-feature separability exists.")
else:
    print("No single feature trivially solves the task.")

## 11. PCA Visualization

In [ ]:
X_eda = df[FEATURES].to_numpy(dtype=np.float32)
y_eda = df[LABEL].to_numpy(dtype=np.int32)

scaler_eda = StandardScaler()
X_scaled_eda = scaler_eda.fit_transform(X_eda)

pca = PCA(n_components=2, random_state=RANDOM_SEED)
X_pca = pca.fit_transform(X_scaled_eda)

pca_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "Label": y_eda
})

fig, ax = plt.subplots(figsize=(8, 6))
for label_value, label_name in [(0, "Normal"), (1, "Ransomware")]:
    subset = pca_df[pca_df["Label"] == label_value]
    ax.scatter(
        subset["PC1"],
        subset["PC2"],
        s=16,
        alpha=0.55,
        label=label_name
    )

ax.set_title(
    f"PCA Projection | Explained variance = "
    f"{100*pca.explained_variance_ratio_.sum():.2f}%"
)
ax.set_xlabel("Principal Component 1")
ax.set_ylabel("Principal Component 2")
ax.legend()
plt.tight_layout()
plt.show()

display(pd.DataFrame({
    "Component": ["PC1", "PC2"],
    "Explained variance ratio": pca.explained_variance_ratio_
}))

# Model Training

From this point onward, the training logic intentionally follows the original pipeline so that EDA does not change the model behavior.

## 12. Stratified Train / Validation / Test Split

In [ ]:
X = df[FEATURES].values.astype(np.float32)
y = df[LABEL].values.astype(np.float32)

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SPLIT,
    stratify=y,
    random_state=RANDOM_SEED
)

val_ratio_adjusted = VAL_SPLIT / (1.0 - TEST_SPLIT)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval,
    y_trainval,
    test_size=val_ratio_adjusted,
    stratify=y_trainval,
    random_state=RANDOM_SEED
)

split_table = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Samples": [len(X_train), len(X_val), len(X_test)],
    "Normal": [
        int((y_train == 0).sum()),
        int((y_val == 0).sum()),
        int((y_test == 0).sum())
    ],
    "Ransomware": [
        int((y_train == 1).sum()),
        int((y_val == 1).sum()),
        int((y_test == 1).sum())
    ]
})

display(split_table)

## 13. Neural Network Architecture

In [ ]:
def build_model(input_dim):
    norm = layers.Normalization(axis=-1)

    model = keras.Sequential([
        keras.Input(shape=(input_dim,), name="sensor_input"),
        norm,

        layers.Dense(
            64,
            activation="relu",
            kernel_regularizer=regularizers.l2(L2_LAMBDA),
            name="dense_1"
        ),
        layers.Dropout(DROPOUT_RATE, name="dropout_1"),

        layers.Dense(
            32,
            activation="relu",
            kernel_regularizer=regularizers.l2(L2_LAMBDA),
            name="dense_2"
        ),
        layers.Dropout(DROPOUT_RATE, name="dropout_2"),

        layers.Dense(1, activation="sigmoid", name="output")
    ], name="ransomware_detector")

    return model, norm

model, norm_layer = build_model(len(FEATURES))

# IMPORTANT: normalization learns statistics from TRAINING DATA ONLY.
norm_layer.adapt(X_train)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.AUC(name="auc"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall")
    ]
)

model.summary()
total_params = model.count_params()

print(f"Total parameters : {total_params:,}")
print(f"Approx. raw parameter storage : {total_params * 4 / 1024:.2f} KB")

## 14. Class Weights

In [ ]:
n_neg = int((y_train == 0).sum())
n_pos = int((y_train == 1).sum())
total = len(y_train)

class_weights = {
    0: (1.0 / n_neg) * (total / 2.0),
    1: (1.0 / n_pos) * (total / 2.0),
}

print("Class weights:")
print(f"Normal     : {class_weights[0]:.6f}")
print(f"Ransomware : {class_weights[1]:.6f}")

## 15. Train Model

In [ ]:
t0 = time.time()

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weights,
    callbacks=[
        callbacks.EarlyStopping(
            monitor="val_auc",
            mode="max",
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=8,
            min_lr=1e-6,
            verbose=1
        ),
        callbacks.ModelCheckpoint(
            "best_checkpoint.keras",
            monitor="val_auc",
            mode="max",
            save_best_only=True,
            verbose=0
        ),
    ],
    verbose=1
)

train_time = time.time() - t0
epochs_run = len(history.history["loss"])

print(f"Training time : {train_time:.2f} s")
print(f"Epochs run    : {epochs_run}")

## 16. Training-Curve Analysis

In [ ]:
h = history.history

# Use the same epoch for train-vs-validation comparison.
best_epoch_idx = int(np.argmax(h["val_auc"]))
best_epoch = best_epoch_idx + 1

train_auc_at_best = h["auc"][best_epoch_idx]
val_auc_at_best = h["val_auc"][best_epoch_idx]
auc_gap = train_auc_at_best - val_auc_at_best

print("Best epoch:", best_epoch)
print(f"Train AUC at best epoch : {train_auc_at_best:.6f}")
print(f"Val AUC at best epoch   : {val_auc_at_best:.6f}")
print(f"AUC gap                 : {auc_gap:.6f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(h["loss"], label="Train loss")
ax.plot(h["val_loss"], label="Validation loss")
ax.axvline(best_epoch_idx, linestyle="--", label=f"Best epoch {best_epoch}")
ax.set_title("Training vs Validation Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Binary cross-entropy")
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(h["auc"], label="Train AUC")
ax.plot(h["val_auc"], label="Validation AUC")
ax.axvline(best_epoch_idx, linestyle="--", label=f"Best epoch {best_epoch}")
ax.set_title("Training vs Validation ROC-AUC")
ax.set_xlabel("Epoch")
ax.set_ylabel("ROC-AUC")
ax.legend()
plt.tight_layout()
plt.show()

## 17. Held-Out Test Evaluation

In [ ]:
y_prob = model.predict(X_test, verbose=0).flatten()

# Same original threshold preserved for deployment consistency.
y_pred = (y_prob >= DECISION_THRESHOLD).astype(int)

roc_auc = roc_auc_score(y_test, y_prob)
avg_pr = average_precision_score(y_test, y_prob)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

metrics_table = pd.DataFrame([{
    "Threshold": DECISION_THRESHOLD,
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "ROC-AUC": roc_auc,
    "PR-AUC": avg_pr,
    "FNR": fn / (fn + tp) if fn + tp else 0,
    "FPR": fp / (fp + tn) if fp + tn else 0,
    "TN": tn,
    "FP": fp,
    "FN": fn,
    "TP": tp,
}])

display(metrics_table)

print(classification_report(
    y_test,
    y_pred,
    target_names=LABEL_NAMES,
    digits=4
))

print("Confusion matrix:")
print(cm)

## 18. ROC & Precision-Recall Curves

In [ ]:
fpr, tpr, roc_thresholds = roc_curve(y_test, y_prob)
pr_precision, pr_recall, pr_thresholds = precision_recall_curve(y_test, y_prob)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.4f}")
ax.plot([0, 1], [0, 1], linestyle="--", label="Random")
ax.set_title("ROC Curve — Held-Out Test Set")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(pr_recall, pr_precision, label=f"PR-AUC = {avg_pr:.4f}")
ax.set_title("Precision-Recall Curve — Held-Out Test Set")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend()
plt.tight_layout()
plt.show()

## 19. Confusion Matrix Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm)

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Normal", "Ransomware"])
ax.set_yticklabels(["Normal", "Ransomware"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix")

for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=14)

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 20. Threshold Sensitivity Analysis

This analysis shows how precision, recall, F1, false positives, and false negatives change with the decision threshold.

**The deployed threshold remains `0.45`** to preserve the original pipeline.

In [ ]:
threshold_grid = np.linspace(0.05, 0.95, 91)
threshold_rows = []

for threshold in threshold_grid:
    pred = (y_prob >= threshold).astype(int)
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_test, pred).ravel()

    threshold_rows.append({
        "Threshold": threshold,
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "FP": fp_t,
        "FN": fn_t
    })

threshold_df = pd.DataFrame(threshold_rows)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(threshold_df["Threshold"], threshold_df["Precision"], label="Precision")
ax.plot(threshold_df["Threshold"], threshold_df["Recall"], label="Recall")
ax.plot(threshold_df["Threshold"], threshold_df["F1"], label="F1")
ax.axvline(
    DECISION_THRESHOLD,
    linestyle="--",
    label=f"Deployed threshold = {DECISION_THRESHOLD}"
)
ax.set_xlabel("Decision Threshold")
ax.set_ylabel("Metric")
ax.set_title("Threshold Sensitivity")
ax.legend()
plt.tight_layout()
plt.show()

display(
    threshold_df.iloc[
        (threshold_df["Threshold"] - DECISION_THRESHOLD).abs().argsort()[:10]
    ].sort_values("Threshold")
)

## 21. Misclassification / Error Analysis

In [ ]:
error_df = pd.DataFrame(X_test, columns=FEATURES)
error_df["Actual"] = y_test.astype(int)
error_df["Probability_Ransomware"] = y_prob
error_df["Predicted"] = y_pred

error_df["ErrorType"] = np.select(
    [
        (error_df["Actual"] == 1) & (error_df["Predicted"] == 0),
        (error_df["Actual"] == 0) & (error_df["Predicted"] == 1),
    ],
    ["False Negative", "False Positive"],
    default="Correct"
)

print(error_df["ErrorType"].value_counts())

print("\nFalse negatives:")
display(
    error_df[error_df["ErrorType"] == "False Negative"]
    .sort_values("Probability_Ransomware")
)

print("\nFalse positives:")
display(
    error_df[error_df["ErrorType"] == "False Positive"]
    .sort_values("Probability_Ransomware", ascending=False)
)

## 22. Five-Fold Stratified Cross-Validation

In [ ]:
kf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_SEED
)

cv_rows = []

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_trainval, y_trainval), start=1):
    tf.keras.backend.clear_session()

    Xtr, Xva = X_trainval[tr_idx], X_trainval[va_idx]
    ytr, yva = y_trainval[tr_idx], y_trainval[va_idx]

    cv_model, cv_norm = build_model(len(FEATURES))
    cv_norm.adapt(Xtr)

    cv_model.compile(
        optimizer=keras.optimizers.Adam(LEARNING_RATE),
        loss="binary_crossentropy",
        metrics=[keras.metrics.AUC(name="auc")]
    )

    cw = {
        0: (1.0 / (ytr == 0).sum()) * (len(ytr) / 2.0),
        1: (1.0 / (ytr == 1).sum()) * (len(ytr) / 2.0),
    }

    cv_model.fit(
        Xtr,
        ytr,
        validation_data=(Xva, yva),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=cw,
        callbacks=[
            callbacks.EarlyStopping(
                monitor="val_auc",
                mode="max",
                patience=PATIENCE,
                restore_best_weights=True,
                verbose=0
            )
        ],
        verbose=0
    )

    cv_prob = cv_model.predict(Xva, verbose=0).flatten()
    fold_auc = roc_auc_score(yva, cv_prob)
    fold_pr_auc = average_precision_score(yva, cv_prob)

    cv_rows.append({
        "Fold": fold,
        "ROC-AUC": fold_auc,
        "PR-AUC": fold_pr_auc
    })

    print(
        f"Fold {fold}: ROC-AUC={fold_auc:.6f}, "
        f"PR-AUC={fold_pr_auc:.6f}"
    )

cv_results = pd.DataFrame(cv_rows)
display(cv_results)

cv_mean = cv_results["ROC-AUC"].mean()
cv_std = cv_results["ROC-AUC"].std(ddof=0)

print(f"Mean CV ROC-AUC = {cv_mean:.6f} ± {cv_std:.6f}")

## 23. Cross-Validation Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(cv_results["Fold"].astype(str), cv_results["ROC-AUC"])
ax.axhline(cv_mean, linestyle="--", label=f"Mean = {cv_mean:.4f}")
ax.set_ylim(max(0, cv_results["ROC-AUC"].min() - 0.05), 1.01)
ax.set_xlabel("Fold")
ax.set_ylabel("ROC-AUC")
ax.set_title("5-Fold Stratified Cross-Validation")
ax.legend()
plt.tight_layout()
plt.show()

# TinyML Deployment

The following sections preserve the original float16 TFLite export and C-header generation.

## 24. Convert to Float16 TensorFlow Lite

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()

with open(MODEL_TFLITE, "wb") as f:
    f.write(tflite_model)

tflite_size = len(tflite_model)

print(f"TFLite model saved: {MODEL_TFLITE}")
print(f"TFLite size       : {tflite_size:,} bytes ({tflite_size/1024:.2f} KB)")

## 25. Full Test-Set Keras ↔ TFLite Parity

In [ ]:
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

keras_prob = model.predict(X_test, verbose=0).flatten()
tflite_prob = np.empty(len(X_test), dtype=np.float32)

for i, sample in enumerate(X_test):
    x = sample.reshape(1, -1).astype(input_details["dtype"])
    interpreter.set_tensor(input_details["index"], x)
    interpreter.invoke()

    tflite_prob[i] = float(
        interpreter.get_tensor(output_details["index"]).ravel()[0]
    )

abs_delta = np.abs(keras_prob - tflite_prob)

keras_class = (keras_prob >= DECISION_THRESHOLD).astype(int)
tflite_class = (tflite_prob >= DECISION_THRESHOLD).astype(int)

parity_table = pd.DataFrame([{
    "Mean absolute probability delta": abs_delta.mean(),
    "Max absolute probability delta": abs_delta.max(),
    "Classification agreement": np.mean(keras_class == tflite_class)
}])

display(parity_table)

## 26. Generate Embedded C Header

In [ ]:
var_name = "ransomware_model"
guard = "RANSOMWARE_MODEL_NEW_H"

hex_lines = []
for i in range(0, len(tflite_model), 12):
    chunk = tflite_model[i:i+12]
    hex_lines.append(
        "    " + ", ".join(f"0x{b:02x}" for b in chunk) + ","
    )

header_str = f'''#ifndef {guard}
#define {guard}

/*
 * Ransomware detection model
 * Architecture: Normalization -> Dense(64) -> Dropout ->
 *               Dense(32) -> Dropout -> Sigmoid
 *
 * Input order:
 *   0 USB_V
 *   1 USB_A
 *   2 USB_W
 *   3 Int_Accel
 *   4 Ext_Accel
 *   5 Net_Traffic
 *   6 RF_Density
 *   7 Sys_Temp
 *
 * Output:
 *   Probability of ransomware in [0, 1]
 *
 * Decision threshold: {DECISION_THRESHOLD}
 */

alignas(8) const unsigned char {var_name}[] = {{
{chr(10).join(hex_lines)}
}};

const unsigned int {var_name}_len = {tflite_size};
const float {var_name}_threshold = {DECISION_THRESHOLD}f;

#endif
'''

with open(MODEL_H_OUT, "w") as f:
    f.write(header_str)

print(f"Saved C header: {MODEL_H_OUT}")

## 27. Save Training Report

In [ ]:
report_lines = [
    "RANSOMWARE DETECTION — TRAINING REPORT",
    "=" * 64,
    f"Dataset: {CSV_FILE}",
    f"Rows: {len(df)}",
    f"Normal samples: {n_normal}",
    f"Ransomware samples: {n_ransom}",
    "",
    "MODEL",
    f"Parameters: {total_params}",
    "Architecture: 8 -> Normalization -> 64 -> 32 -> 1",
    f"Dropout: {DROPOUT_RATE}",
    f"L2: {L2_LAMBDA}",
    f"Learning rate: {LEARNING_RATE}",
    f"Decision threshold: {DECISION_THRESHOLD}",
    "",
    "TEST RESULTS",
    f"Accuracy: {accuracy:.6f}",
    f"Precision: {precision:.6f}",
    f"Recall: {recall:.6f}",
    f"F1: {f1:.6f}",
    f"ROC-AUC: {roc_auc:.6f}",
    f"PR-AUC: {avg_pr:.6f}",
    f"TN={tn}, FP={fp}, FN={fn}, TP={tp}",
    "",
    "CROSS VALIDATION",
    f"Mean ROC-AUC: {cv_mean:.6f}",
    f"Std ROC-AUC: {cv_std:.6f}",
    "",
    "DATASET DIAGNOSTIC",
    f"Best single-feature ROC-AUC: {best_single_auc:.6f}",
    "",
    "TFLITE",
    f"Size bytes: {tflite_size}",
    f"Mean Keras/TFLite delta: {abs_delta.mean():.8f}",
    f"Max Keras/TFLite delta: {abs_delta.max():.8f}",
    f"Class agreement: {np.mean(keras_class == tflite_class):.6f}",
]

report = "\n".join(report_lines)

Path(REPORT_FILE).write_text(report)
print(report)
print(f"\nSaved: {REPORT_FILE}")

## 28. Final Interpretation

Use the following points when presenting the project:

1. **Dataset integrity**  
   Verify missing values, duplicates, class balance, and feature distributions before training.

2. **No preprocessing leakage**  
   The TensorFlow normalization layer is adapted only on the training partition.

3. **Stratified evaluation**  
   Train, validation, and held-out test sets preserve class proportions.

4. **Robust evaluation**  
   Report ROC-AUC, PR-AUC, precision, recall, F1, confusion matrix, and five-fold CV.

5. **Dataset limitation**  
   If single-feature ROC-AUC is close to `1.0`, the current dataset is strongly separable. The high model performance should therefore be described as validation of the detection/deployment pipeline rather than proof of real-world generalization.

6. **Deployment validation**  
   Keras and float16 TFLite outputs are checked across the full held-out test set before exporting the C header.

### Recommended resume phrasing

> Developed an 8-feature hardware-telemetry ransomware detection pipeline with stratified validation, 5-fold cross-validation, comprehensive EDA and error analysis; converted the trained neural classifier to float16 TensorFlow Lite and C-compatible firmware representation for embedded deployment.

Avoid claiming that the current results prove universal real-world ransomware detection unless the model is independently evaluated on external real-world telemetry.